In [ ]:
!pip install weaviate-client requests --quiet

# Environment
In order for this to run, both the client and server must have access to Keycloack from a single url.
For a Mac with Rancher Desktop, for example, you can add this to your /etc/hosts

```
127.0.0.1 	host.docker.internal
```

First, run only keycloack, otherwise Weaviate will not be up due to no helm created

In [31]:
!docker compose up -d keycloack

/opt/homebrew/Cellar/python@3.13/3.13.12_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=84102) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


no such service: keycloack


Run this idempotent code. It will:
Configures:
  1. Realm 'weaviate'
  2. Public client 'weaviate' with Direct Access Grants + Audience mapper
     + Group Membership mapper (groups appear in JWT as 'groups' claim)
  3. User profile: removes firstName/lastName as required (Keycloak 26 quirk)
  4. User 'admin-user' (password: test)  — Weaviate RBAC root user
  5. Group 'SpecialCollections'
  6. User 'newuser' (password: newuser)  — member of SpecialCollections
  7. Create group 'SpecialCollections'
  8. Create newuser and add to SpecialCollections

In [ ]:
import requests

KEYCLOAK   = "http://host.docker.internal:8081"
REALM      = "weaviate"
CLIENT_ID  = "weaviate"
ADMIN_USER = "admin-user"
ADMIN_PASS = "test"
NEW_USER   = "newuser"
NEW_PASS   = "newuser"
GROUP_NAME = "SpecialCollections"

def admin_token():
    r = requests.post(
        f"{KEYCLOAK}/realms/master/protocol/openid-connect/token",
        data={"grant_type": "password", "client_id": "admin-cli",
              "username": "admin", "password": "admin"},
    )
    r.raise_for_status()
    return r.json()["access_token"]

def adm(method, path, **kw):
    return requests.request(
        method, f"{KEYCLOAK}/admin/{path}",
        headers={"Authorization": f"Bearer {admin_token()}",
                 "Content-Type": "application/json"},
        **kw,
    )

# 1. Create realm
r = adm("POST", "realms", json={"realm": REALM, "enabled": True})
print(f"[1] Create realm:            {r.status_code}  (409 = already exists, ok)")

# 2. Remove firstName / lastName / email as required in User Profile.
#    Keycloak 26 marks them required for the 'user' role by default —
#    this causes "Account is not fully set up" on Resource Owner Password login.
profile = adm("GET", f"realms/{REALM}/users/profile").json()
for attr in profile.get("attributes", []):
    if attr.get("name") in ("firstName", "lastName", "email"):
        attr.pop("required", None)
r = adm("PUT", f"realms/{REALM}/users/profile", json=profile)
print(f"[2] Patch user profile:      {r.status_code}")

# 3. Create public client with Direct Access Grants enabled
r = adm("POST", f"realms/{REALM}/clients", json={
    "clientId": CLIENT_ID,
    "enabled": True,
    "publicClient": True,
    "directAccessGrantsEnabled": True,
    "protocol": "openid-connect",
})
print(f"[3] Create client:           {r.status_code}  (409 = already exists, ok)")

client_uuid = adm("GET", f"realms/{REALM}/clients?clientId={CLIENT_ID}").json()[0]["id"]

# 4. Add Audience mapper
r = adm("POST", f"realms/{REALM}/clients/{client_uuid}/protocol-mappers/models", json={
    "name": "weaviate-audience",
    "protocol": "openid-connect",
    "protocolMapper": "oidc-audience-mapper",
    "config": {
        "included.client.audience": CLIENT_ID,
        "id.token.claim": "false",
        "access.token.claim": "true",
    },
})
print(f"[4] Add audience mapper:     {r.status_code}  (409 = already exists, ok)")

# 5. Add Group Membership mapper (full.path=true → '/SpecialCollections' in JWT)
r = adm("POST", f"realms/{REALM}/clients/{client_uuid}/protocol-mappers/models", json={
    "name": "weaviate-groups",
    "protocol": "openid-connect",
    "protocolMapper": "oidc-group-membership-mapper",
    "config": {
        "claim.name": "groups",
        "full.path": "true",
        "id.token.claim": "false",
        "access.token.claim": "true",
        "userinfo.token.claim": "false",
    },
})
print(f"[5] Add groups mapper:       {r.status_code}  (409 = already exists, ok)")

# 6. Create admin user
r = adm("POST", f"realms/{REALM}/users", json={
    "username": ADMIN_USER,
    "enabled": True,
    "emailVerified": True,
    "credentials": [{"type": "password", "value": ADMIN_PASS, "temporary": False}],
})
print(f"[6] Create admin-user:       {r.status_code}  (409 = already exists, ok)")

# 7. Create group 'SpecialCollections'
r = adm("POST", f"realms/{REALM}/groups", json={"name": GROUP_NAME})
print(f"[7] Create group:            {r.status_code}  (409 = already exists, ok)")
group_id = adm("GET", f"realms/{REALM}/groups?search={GROUP_NAME}").json()[0]["id"]
print(f"    Group UUID: {group_id}")

# 8. Create newuser and add to SpecialCollections
r = adm("POST", f"realms/{REALM}/users", json={
    "username": NEW_USER,
    "enabled": True,
    "emailVerified": True,
    "credentials": [{"type": "password", "value": NEW_PASS, "temporary": False}],
})
print(f"[8] Create newuser:          {r.status_code}  (409 = already exists, ok)")
new_user_id = adm("GET", f"realms/{REALM}/users?username={NEW_USER}").json()[0]["id"]

r = adm("PUT", f"realms/{REALM}/users/{new_user_id}/groups/{group_id}")
print(f"    Add newuser to group:    {r.status_code}")

print("\nKeycloak ready.")

[1] Create realm:            201  (409 = already exists, ok)
[2] Patch user profile:      200
[3] Create client:           201  (409 = already exists, ok)
[4] Add audience mapper:     201  (409 = already exists, ok)
[5] Add groups mapper:       201  (409 = already exists, ok)
[6] Create admin-user:       201  (409 = already exists, ok)
[7] Create group:            201  (409 = already exists, ok)
    Group UUID: bf92cda6-be7a-4e52-8840-fbec5757bbac
[8] Create newuser:          201  (409 = already exists, ok)
    Add newuser to group:    204

Keycloak ready.


# Now, Run Weaviate
We will connect as the admin user set roles and group setup, then create some collections.

In [30]:
!docker compose up -d weaviate

/opt/homebrew/Cellar/python@3.13/3.13.12_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=84102) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


WARN[0000] No services to build                         
[+] up 1/1
 ✔ Container keycloack-keycloak-1 Running                                  0.0s 
[+] up 1/1
 ✔ Container keycloack-keycloak-1 Running                                  0.0s 
[+] up 1/1
 ✔ Container keycloack-keycloak-1 Running                                  0.0s 


Now let's connect with admin user, in order to create the RBAC roles and groups

In [ ]:
import weaviate
from weaviate.classes.init import Auth

# Weaviate is on localhost:8080, Keycloak is on localhost:8081
# Auth.client_password uses the OIDC Resource Owner Password flow.
# The token is fetched from Keycloak using the credentials below,
# then passed as a Bearer token to Weaviate.
# this means both client and server must have acess to OIDC server (Keycloak)
# for token fetching and verification.
client = weaviate.connect_to_local(
    auth_credentials=Auth.client_password(
        username="admin-user",
        password="test"
    )
)

print(client.is_ready())

True


Exception in thread TokenRefresh:
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.12_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.12_1/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dudanogueira/dev/weaviate/lab/keycloack/.venv/lib/python3.13/site-packages/weaviate/connect/v4.py", line 585, in periodic_refresh_token
    refresh_token()
    ~~~~~~~~~~~~~^^
  File "/Users/dudanogueira/dev/weaviate/lab/keycloack/.venv/lib/python3.13/site-packages/weaviate/connect/v4.py", line 551, in refresh_token
    self._client.token = self._client.refresh_token(
                         ~~~~~~~~~~~~~~~~~~~~~~~~~~^
        url=self._client.metadata["token_endpoint"]
        ^^^^^^^^^^^^^^^^

In [33]:
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.rbac import Permissions

# Connect as admin-user (RBAC root) to manage roles
admin_client = weaviate.connect_to_local(
    auth_credentials=Auth.client_password(
        username="admin-user", password="test"
    ),
)

ROLE_NAME  = "special-collections-role"
GROUP_NAME = "/SpecialCollections"  # Keycloak full path — matches JWT 'groups' claim

# Create a role with read access to all collections
try:
    admin_client.roles.create(
        role_name=ROLE_NAME,
        permissions=[
            Permissions.collections(collection="Special*", read_config=True),
            Permissions.data(collection="Special*", read=True),
        ],
    )
    print(f"Role '{ROLE_NAME}' created.")
except Exception as e:
    print(f"Role create: {e}")

# Assign the role to the group
try:
    admin_client.groups.oidc.assign_roles(group_id=GROUP_NAME, role_names=[ROLE_NAME])
    print(f"Role '{ROLE_NAME}' assigned to group '{GROUP_NAME}'.")
except Exception as e:
    print(f"Assign error: {e}")

# Verify
try:
    role = admin_client.roles.get(ROLE_NAME)
    print("\nRole details:", role)
except Exception as e:
    print(f"Verify error: {e}")

# Let's create some collections
for col in ["SpecialCollection1", "SpecialCollection2", "NotSpecialCollection1"]:
    try:
        admin_client.collections.create(col)
        print(f"Collection '{col}' created.")
    except Exception as e:
        print(f"Collection create error for '{col}': {e}")

admin_client.close()


Role 'special-collections-role' created.
Role 'special-collections-role' assigned to group '/SpecialCollections'.

Role details: Role(name='special-collections-role', alias_permissions=[], cluster_permissions=[], collections_permissions=[CollectionsPermissionOutput(actions={<CollectionsAction.READ: 'read_collections'>}, collection='Special*')], data_permissions=[DataPermissionOutput(actions={<DataAction.READ: 'read_data'>}, collection='Special*', tenant='*')], roles_permissions=[], users_permissions=[], backups_permissions=[], nodes_permissions=[], tenants_permissions=[], replicate_permissions=[], groups_permissions=[])
Collection 'SpecialCollection1' created.
Collection 'SpecialCollection2' created.
Collection 'NotSpecialCollection1' created.


In [35]:
# now connect with our user
client = weaviate.connect_to_local(
    auth_credentials=Auth.client_password(
        username="newuser",
        password="newuser"
    )
)
collection_keys = client.collections.list_all().keys()
print("\nCollections visible to newuser:", collection_keys)
assert "NotSpecialCollection1" not in collection_keys, "newuser should NOT see NotSpecialCollection1"
assert "SpecialCollection1" in collection_keys, "newuser should see SpecialCollection1"
assert "SpecialCollection2" in collection_keys, "newuser should see SpecialCollection2"


Collections visible to newuser: dict_keys(['SpecialCollection1', 'SpecialCollection2'])


# TLS / Self-Signed Certificate Test

This section tests Weaviate connectivity to a **HTTPS Keycloak** that uses a **self-signed certificate**.

Key things being validated:
- Keycloak is configured with a self-signed TLS cert for `host.docker.internal`
- Weaviate trusts that cert via `AUTHENTICATION_OIDC_CERTIFICATE` (inline PEM)
- OIDC token flow and RBAC still work correctly end-to-end

See PR [weaviate#10813](https://github.com/weaviate/weaviate/pull/10813) for the `AUTHENTICATION_OIDC_SKIP_TLS_VERIFY` alternative.

**Prerequisites:** `openssl` must be available on the host (`which openssl`).

In [ ]:
# Step 1: Generate self-signed CA + Keycloak server certificate.
# Safe to re-run — overwrites existing certs in ./certs/
import os, subprocess

os.makedirs("certs", exist_ok=True)
result = subprocess.run(["bash", "certs/generate.sh"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Certificate generation failed")

In [ ]:
# Step 2: Tear down any previous TLS stack, then start the TLS stack.
# The CA PEM is injected into the shell environment so docker-compose can
# substitute ${AUTHENTICATION_OIDC_CERTIFICATE} in docker-compose-tls.yml.
import os, subprocess

ca_pem = open("certs/ca.pem").read()
env = {**os.environ, "AUTHENTICATION_OIDC_CERTIFICATE": ca_pem}

# Tear down cleanly first (idempotent)
subprocess.run(
    ["docker", "compose", "-f", "docker-compose-tls.yml", "--project-name", "weaviate-tls", "down", "-v"],
    env=env, capture_output=True,
)

result = subprocess.run(
    ["docker", "compose", "-f", "docker-compose-tls.yml", "--project-name", "weaviate-tls", "up", "-d"],
    env=env, capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("docker compose up failed")

In [ ]:
# Step 3: Wait for Keycloak HTTPS and Weaviate to be ready.
import time, requests

CA_CERT    = "certs/ca.pem"
KC_HTTPS   = "https://host.docker.internal:8443"
KC_HTTP    = "http://host.docker.internal:8081"   # HTTP admin port (always on in start-dev)
WV_HTTP    = "http://localhost:8080"

print("Waiting for Keycloak HTTPS...")
for i in range(40):
    try:
        r = requests.get(f"{KC_HTTPS}/realms/master", verify=CA_CERT, timeout=3)
        if r.status_code < 500:
            print(f"  Keycloak HTTPS ready (HTTP {r.status_code})")
            break
    except Exception:
        pass
    time.sleep(3)
    print(f"  attempt {i+1}...")
else:
    raise TimeoutError("Keycloak HTTPS did not become ready")

print("Waiting for Weaviate...")
for i in range(40):
    try:
        r = requests.get(f"{WV_HTTP}/v1/.well-known/ready", timeout=3)
        if r.status_code == 200:
            print("  Weaviate ready.")
            break
    except Exception:
        pass
    time.sleep(3)
    print(f"  attempt {i+1}...")
else:
    raise TimeoutError("Weaviate did not become ready")

In [ ]:
# Step 4: Configure Keycloak (idempotent).
# Admin API calls use the HTTP port (8081 → 8080 inside Keycloak).
# This avoids CA cert complexity for admin setup while keeping OIDC on HTTPS.
import requests

KEYCLOAK   = "http://host.docker.internal:8081"
REALM      = "weaviate"
CLIENT_ID  = "weaviate"
ADMIN_USER = "admin-user"
ADMIN_PASS = "test"
NEW_USER   = "newuser"
NEW_PASS   = "newuser"
GROUP_NAME = "SpecialCollections"

def admin_token():
    r = requests.post(
        f"{KEYCLOAK}/realms/master/protocol/openid-connect/token",
        data={"grant_type": "password", "client_id": "admin-cli",
              "username": "admin", "password": "admin"},
    )
    r.raise_for_status()
    return r.json()["access_token"]

def adm(method, path, **kw):
    return requests.request(
        method, f"{KEYCLOAK}/admin/{path}",
        headers={"Authorization": f"Bearer {admin_token()}",
                 "Content-Type": "application/json"},
        **kw,
    )

r = adm("POST", "realms", json={"realm": REALM, "enabled": True})
print(f"[1] Create realm:            {r.status_code}  (409 = already exists, ok)")

profile = adm("GET", f"realms/{REALM}/users/profile").json()
for attr in profile.get("attributes", []):
    if attr.get("name") in ("firstName", "lastName", "email"):
        attr.pop("required", None)
r = adm("PUT", f"realms/{REALM}/users/profile", json=profile)
print(f"[2] Patch user profile:      {r.status_code}")

r = adm("POST", f"realms/{REALM}/clients", json={
    "clientId": CLIENT_ID, "enabled": True,
    "publicClient": True, "directAccessGrantsEnabled": True,
    "protocol": "openid-connect",
})
print(f"[3] Create client:           {r.status_code}  (409 = already exists, ok)")

client_uuid = adm("GET", f"realms/{REALM}/clients?clientId={CLIENT_ID}").json()[0]["id"]

r = adm("POST", f"realms/{REALM}/clients/{client_uuid}/protocol-mappers/models", json={
    "name": "weaviate-audience", "protocol": "openid-connect",
    "protocolMapper": "oidc-audience-mapper",
    "config": {"included.client.audience": CLIENT_ID,
               "id.token.claim": "false", "access.token.claim": "true"},
})
print(f"[4] Add audience mapper:     {r.status_code}  (409 = already exists, ok)")

r = adm("POST", f"realms/{REALM}/clients/{client_uuid}/protocol-mappers/models", json={
    "name": "weaviate-groups", "protocol": "openid-connect",
    "protocolMapper": "oidc-group-membership-mapper",
    "config": {"claim.name": "groups", "full.path": "true",
               "id.token.claim": "false", "access.token.claim": "true",
               "userinfo.token.claim": "false"},
})
print(f"[5] Add groups mapper:       {r.status_code}  (409 = already exists, ok)")

r = adm("POST", f"realms/{REALM}/users", json={
    "username": ADMIN_USER, "enabled": True, "emailVerified": True,
    "credentials": [{"type": "password", "value": ADMIN_PASS, "temporary": False}],
})
print(f"[6] Create admin-user:       {r.status_code}  (409 = already exists, ok)")

r = adm("POST", f"realms/{REALM}/groups", json={"name": GROUP_NAME})
print(f"[7] Create group:            {r.status_code}  (409 = already exists, ok)")
group_id = adm("GET", f"realms/{REALM}/groups?search={GROUP_NAME}").json()[0]["id"]

r = adm("POST", f"realms/{REALM}/users", json={
    "username": NEW_USER, "enabled": True, "emailVerified": True,
    "credentials": [{"type": "password", "value": NEW_PASS, "temporary": False}],
})
print(f"[8] Create newuser:          {r.status_code}  (409 = already exists, ok)")
new_user_id = adm("GET", f"realms/{REALM}/users?username={NEW_USER}").json()[0]["id"]
r = adm("PUT", f"realms/{REALM}/users/{new_user_id}/groups/{group_id}")
print(f"    Add newuser to group:    {r.status_code}")

print("\nKeycloak ready.")

In [ ]:
# Step 5: Set up RBAC roles and collections via the API key (no OIDC needed for this).
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.rbac import Permissions

admin_client = weaviate.connect_to_local(
    auth_credentials=Auth.api_key("root-user-key"),
)

ROLE_NAME      = "special-collections-role"
KC_GROUP_CLAIM = "/SpecialCollections"  # full.path=true → leading slash

try:
    admin_client.roles.create(
        role_name=ROLE_NAME,
        permissions=[
            Permissions.collections(collection="Special*", read_config=True),
            Permissions.data(collection="Special*", read=True),
        ],
    )
    print(f"Role '{ROLE_NAME}' created.")
except Exception as e:
    print(f"Role create: {e}")

try:
    admin_client.groups.oidc.assign_roles(group_id=KC_GROUP_CLAIM, role_names=[ROLE_NAME])
    print(f"Role assigned to group '{KC_GROUP_CLAIM}'.")
except Exception as e:
    print(f"Assign error: {e}")

for col in ["SpecialCollection1", "SpecialCollection2", "NotSpecialCollection1"]:
    try:
        admin_client.collections.create(col)
        print(f"Collection '{col}' created.")
    except Exception as e:
        print(f"Collection '{col}': {e}")

admin_client.close()

In [ ]:
# Step 6: Negative test — confirm the self-signed cert is NOT trusted by default.
# This is why AUTHENTICATION_OIDC_CERTIFICATE exists: without it Weaviate (and Python)
# would reject Keycloak's TLS handshake.
import requests

KC_HTTPS = "https://host.docker.internal:8443"

try:
    requests.get(f"{KC_HTTPS}/realms/weaviate", verify=True, timeout=5)
    print("UNEXPECTED: request succeeded without CA cert (should have failed!)")
except requests.exceptions.SSLError as e:
    print("Expected SSLError — self-signed cert is not in the default trust store.")
    print(f"  {type(e).__name__}: {str(e)[:120]}")
    print()
    print("=> This is the exact error Weaviate would get without AUTHENTICATION_OIDC_CERTIFICATE.")

# Now confirm it succeeds with our CA cert
r = requests.get(f"{KC_HTTPS}/realms/weaviate", verify="certs/ca.pem", timeout=5)
print(f"\nWith CA cert: HTTP {r.status_code} — TLS verified successfully.")

In [ ]:
# Step 7: Fetch a token from Keycloak over HTTPS (using our CA cert),
# then pass it to Weaviate as a Bearer token.
# Weaviate validates the token by calling Keycloak's JWKS endpoint over HTTPS
# using the CA cert set in AUTHENTICATION_OIDC_CERTIFICATE — that's what we're testing.
import requests, base64, json, weaviate
from weaviate.classes.init import Auth

CA_CERT  = "certs/ca.pem"
KC_HTTPS = "https://host.docker.internal:8443"
REALM    = "weaviate"

def get_oidc_token(username, password):
    r = requests.post(
        f"{KC_HTTPS}/realms/{REALM}/protocol/openid-connect/token",
        data={"grant_type": "password", "client_id": "weaviate",
              "username": username, "password": password},
        verify=CA_CERT,
    )
    r.raise_for_status()
    return r.json()["access_token"]

def decode_claims(token):
    payload = token.split(".")[1]
    payload += "=" * (4 - len(payload) % 4)
    return json.loads(base64.b64decode(payload))

# -- admin-user --
admin_token = get_oidc_token("admin-user", "test")
claims = decode_claims(admin_token)
print("admin-user token claims:")
print(f"  iss:                {claims.get('iss')}")
print(f"  preferred_username: {claims.get('preferred_username')}")
print(f"  aud:                {claims.get('aud')}")
print(f"  groups:             {claims.get('groups')}")

# Connect to Weaviate with the HTTPS-issued token.
# Weaviate validates it against https://host.docker.internal:8443 using the CA cert.
client = weaviate.connect_to_local(
    auth_credentials=Auth.bearer_token(access_token=admin_token),
)
print(f"\nWeaviate ready: {client.is_ready()}")
collections = list(client.collections.list_all().keys())
print(f"All collections (admin view): {collections}")
client.close()

In [ ]:
# Step 8: Verify RBAC still works correctly via HTTPS tokens.
# newuser is in /SpecialCollections and should only see Special* collections.
newuser_token = get_oidc_token("newuser", "newuser")
newuser_claims = decode_claims(newuser_token)
print("newuser groups claim:", newuser_claims.get("groups"))

client = weaviate.connect_to_local(
    auth_credentials=Auth.bearer_token(access_token=newuser_token),
)
collection_keys = set(client.collections.list_all().keys())
print(f"\nCollections visible to newuser: {collection_keys}")

assert "SpecialCollection1"    in collection_keys, "newuser should see SpecialCollection1"
assert "SpecialCollection2"    in collection_keys, "newuser should see SpecialCollection2"
assert "NotSpecialCollection1" not in collection_keys, "newuser should NOT see NotSpecialCollection1"

print("\nAll RBAC assertions passed — TLS + self-signed cert + RBAC working correctly.")
client.close()

In [ ]:
# Step 9 (optional): Tear down the TLS stack and remove volumes.
import subprocess
result = subprocess.run(
    ["docker", "compose", "-f", "docker-compose-tls.yml", "--project-name", "weaviate-tls", "down", "-v"],
    capture_output=True, text=True,
)
print(result.stdout or result.stderr)